# Lecture 2 — Temperature: from weather to climate

## Mediterrenean: analysing a real ERA5 monthly temperature record

In Lecture 1 we introduced the distinction between **weather** and **climate** and learned some basic statistical concepts.

Here we apply those ideas to a real climate dataset: **monthly ERA5 2-m air temperature**.

The dataset has already been spatially selected/aggregated for the selected area, so we can work directly with the resulting regional time series.

We will investigate:

- the seasonal cycle
- climatology
- temperature anomalies
- interannual variability
- the distribution of annual temperatures
- warmest and coldest years
- long-term trends
- low-frequency variability

> **Important:** this is monthly data. We are not calculating hourly or daily extremes.

## 1. Load the data

The course dataset is in:

```text
climate-course/
├── environment.yml
├── notebooks/
│   └── 02_temperature.ipynb
└── data/
    └── era5_monthly_t2m_med.nc
```

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import linregress

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12

data_file = Path("../data/era5_monthly_t2m_med.nc")

if not data_file.exists():
    raise FileNotFoundError(
        "Course dataset not found. Make sure "
        "*.nc is in ../data/."
    )

ds = xr.open_dataset(data_file)

ds

## 2. Inspect the dataset

Before analysing a climate dataset, always check:

- dimensions
- coordinates
- variables
- units
- time coverage

This prevents us from making assumptions about the data.

In [ ]:
print("Dimensions:")
print(ds.dims)

print("\nCoordinates:")
print(list(ds.coords))

print("\nVariables:")
print(list(ds.data_vars))

print("\nTemperature variable:")
print(ds["t2m"])

print("\nUnits:", ds["t2m"].attrs.get("units"))

print("\nTime coverage:")
print(ds.time.values[0], "to", ds.time.values[-1])

## 3. Temperature in degrees Celsius

ERA5 temperature is stored in Kelvin.

We convert it to degrees Celsius:

\[
T_{°C} = T_K - 273.15
\]

We create a new variable so that the original data remain unchanged.

In [ ]:
temperature = ds["t2m"] - 273.15
temperature.name = "temperature"
temperature.attrs["units"] = "°C"

temperature

## 4. Plot the monthly temperature record

Let's first look at the complete time series.

What features can you identify?

- seasonal cycle
- year-to-year variability
- long-term changes

In [ ]:
temperature.plot()

plt.title("Italy monthly mean temperature")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

## 5. The seasonal cycle

Temperature has a strong and relatively predictable annual cycle.

We can estimate the typical temperature for each calendar month by averaging all Januaries together, all Februaries together, and so on.

In [ ]:
monthly_climatology = temperature.groupby("time.month").mean()

months = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

plt.plot(months, monthly_climatology.values, marker="o")

plt.title("Italy: mean seasonal cycle")
plt.xlabel("Month")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

### Question

Why is the seasonal cycle so large compared with the temperature difference between many neighbouring years?

What physical processes produce the annual cycle of temperature?

## 6. Climate climatology: 1991–2020

A **climatology** is a statistical description of typical climate conditions over a specified reference period.

We use **1991–2020** and calculate a mean temperature for each calendar month.

In [ ]:
reference = temperature.sel(
    time=slice("1991-01-01", "2020-12-31")
)

climatology_1991_2020 = reference.groupby("time.month").mean()

plt.plot(months, climatology_1991_2020.values, marker="o")

plt.title("Italy temperature climatology, 1991–2020")
plt.xlabel("Month")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

## 7. Temperature anomalies

Absolute temperature is strongly affected by the season.

To identify unusually warm or cold conditions, we calculate an **anomaly** relative to the 1991–2020 climatology:

\[
T'(t) = T(t) - \overline{T}_{1991-2020}
\]

For January, for example, we compare January temperatures with the typical January temperature — not with the annual mean.

In [ ]:
climatology_for_each_month = climatology_1991_2020.sel(
    month=temperature.time.dt.month
)

anomalies = temperature - climatology_for_each_month
anomalies.name = "temperature_anomaly"

anomalies.plot()

plt.axhline(0, linewidth=1)
plt.title("Italy monthly temperature anomalies")
plt.xlabel("Year")
plt.ylabel("Temperature anomaly (°C)")
plt.grid(alpha=0.25)
plt.show()

### Interpretation

An anomaly of +2°C does **not** mean that Italy was 2°C warmer than another country.

It means that the temperature was 2°C above the temperature that is typical for that calendar month during the chosen reference period.

## 8. Annual mean temperature

The seasonal cycle can make long-term changes harder to see.

We therefore calculate one mean temperature for each year.

In [ ]:
annual_temperature = temperature.groupby("time.year").mean()

annual_temperature

In [ ]:
plt.plot(
    annual_temperature.year,
    annual_temperature,
    marker="o",
    markersize=3,
    linewidth=1
)

plt.title("Italy annual mean temperature")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

## 9. Interannual variability

The annual temperature does not increase smoothly.

Even in a warming climate, individual years can be cooler or warmer than the long-term trend.

Let's look at the distribution of annual mean temperatures.

In [ ]:
plt.hist(
    annual_temperature.values,
    bins=15,
    edgecolor="black"
)

plt.xlabel("Annual mean temperature (°C)")
plt.ylabel("Number of years")
plt.title("Distribution of Italy annual mean temperature")
plt.show()

## 10. Warmest and coldest years

Which years were the warmest and coldest in the record?

This is one way to connect a statistical distribution to actual climate events.

In [ ]:
annual_df = annual_temperature.to_dataframe(
    name="temperature"
)

warmest = annual_df.sort_values(
    "temperature", ascending=False
).head(5)

coldest = annual_df.sort_values(
    "temperature"
).head(5)

print("Five warmest years:")
display(warmest)

print("\nFive coldest years:")
display(coldest)

## 11. Long-term temperature trend

A simple first estimate of climate change is a linear trend:

\[
T(t) = a + bt
\]

where the slope \(b\) describes the rate of temperature change.

We report the trend in **°C per decade**.

In [ ]:
years = annual_temperature.year.values
values = annual_temperature.values

slope, intercept, r_value, p_value, std_err = linregress(
    years, values
)

trend_per_decade = slope * 10

print(f"Linear trend: {trend_per_decade:.2f} °C per decade")
print(f"R² = {r_value**2:.2f}")
print(f"p-value = {p_value:.3g}")

In [ ]:
trend_line = intercept + slope * years

plt.plot(years, values, label="Annual mean")
plt.plot(
    years,
    trend_line,
    linewidth=2,
    label="Linear trend"
)

plt.title("Italy long-term temperature trend")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### Important

The linear trend is a **statistical description of the long-term change**.

It is not a prediction that every future year will be warmer than the previous year.

The deviations around the trend represent variability on shorter timescales.

## 12. Long-term signal versus variability

An 11-year running mean suppresses much of the year-to-year variability and makes lower-frequency changes easier to see.

In [ ]:
running_mean = annual_temperature.rolling(
    year=11,
    center=True
).mean()

plt.plot(
    annual_temperature.year,
    annual_temperature,
    alpha=0.4,
    label="Annual mean"
)

plt.plot(
    running_mean.year,
    running_mean,
    linewidth=2,
    label="11-year running mean"
)

plt.title("Italy temperature: annual variability and low-frequency signal")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

# Exercises

### Exercise 1 — Change the reference period

Calculate anomalies relative to **1961–1990** instead of 1991–2020.

- How do the anomaly values change?
- Does the underlying trend change?

---

### Exercise 2 — Compare periods

Calculate the mean annual temperature for:

- 1951–1980
- 1981–2010
- 1991–2020

How much warmer is each period than the previous one?

---

### Exercise 3 — Trend sensitivity

Calculate the temperature trend separately for:

- 1950–2025
- 1980–2025
- 2000–2025

Is the estimated trend identical?

Why might estimates over shorter periods be more sensitive to variability?

---

### Exercise 4 — Climate versus weather

An individual year can be colder than the previous year even when the long-term trend is positive.

Explain why this is not contradictory.

Use the concepts:

- climate trend
- interannual variability
- seasonal cycle
- anomalies

# Key takeaways

We have followed a complete climate-analysis workflow:

**monthly temperature → seasonal cycle → climatology → anomalies → annual means → variability → distribution → trend**

The central idea is:

> **Climate is not simply the average temperature. It includes the mean state, variability, distributions, extremes, and their changes over time.**

This is the foundation for the more detailed climate-system analysis in the following lectures.